# Experimento 04: LightGBM (Gradient Boosting de Alta Eficiencia)

Tras confirmar empíricamente la lentitud de XGBoost (arquitectura *level-wise*) en espacios vectoriales NLP altamente dispersos, evaluamos la alternativa de la industria: **LightGBM**.
Este algoritmo de Microsoft utiliza un crecimiento de árboles *leaf-wise*, minimizando el cálculo de ramas inútiles y optimizando el consumo de CPU en matrices llenas de ceros.

### Paso 1: Carga Express y Preparación (Label Encoding)
Al igual que XGBoost, LightGBM está escrito en C++ y colapsa si la columna objetivo contiene cadenas de texto (`Service Outages`). Por ello, cargamos las matrices y transformamos inmediatamente las colas de soporte a formato numérico (0 al 6) mediante `LabelEncoder`.

In [1]:
import scipy.sparse
import pandas as pd
import time
from sklearn.preprocessing import LabelEncoder

print("Iniciando carga de matrices en memoria (Modo MLOps)...")
start_load = time.time()

# 1. CARGA INGLÉS
en_X_train = scipy.sparse.load_npz("../data/features/en_X_train_tfidf.npz")
en_X_val = scipy.sparse.load_npz("../data/features/en_X_val_tfidf.npz")

en_y_train = pd.read_csv("../data/features/en_y_train.csv").iloc[:, 0]
en_y_val = pd.read_csv("../data/features/en_y_val.csv").iloc[:, 0]

# 2. CARGA ESPAÑOL
es_X_train = scipy.sparse.load_npz("../data/features/es_X_train_tfidf.npz")
es_X_val = scipy.sparse.load_npz("../data/features/es_X_val_tfidf.npz")

es_y_train = pd.read_csv("../data/features/es_y_train.csv").iloc[:, 0]
es_y_val = pd.read_csv("../data/features/es_y_val.csv").iloc[:, 0]

# 3. CODIFICADOR DE ETIQUETAS (Obligatorio para Boosting)
le = LabelEncoder()
en_y_train_enc = le.fit_transform(en_y_train)
en_y_val_enc = le.transform(en_y_val)

es_y_train_enc = le.transform(es_y_train)
es_y_val_enc = le.transform(es_y_val)

print(f"✅ Matrices y Etiquetas (Numéricas) cargadas en {round(time.time() - start_load, 2)} segundos.")

Iniciando carga de matrices en memoria (Modo MLOps)...
✅ Matrices y Etiquetas (Numéricas) cargadas en 0.15 segundos.


### Paso 2: Experimento Control (LightGBM con Pesos de Clase)

Vamos a evaluar si la arquitectura de crecimiento "por hojas" (*leaf-wise*) de Microsoft soluciona el colapso de lentitud que sufrió XGBoost. 
Además, aplicamos *Cost-Sensitive Learning* (Pesos de Clase). A diferencia de XGBoost, LightGBM acepta nativamente el parámetro `class_weight='balanced'`, encargándose internamente de penalizar de forma drástica los fallos en las clases minoritarias (averías) durante el cálculo del gradiente.

In [2]:
from sklearn.metrics import classification_report, f1_score
import lightgbm as lgb
import time

# FUNCIÓN FÁBRICA PARA LIGHTGBM
def train_evaluate_lgbm_weights(X_train, y_train_enc, X_val, y_val_enc, label_encoder, exp_name):
    print(f"\n=======================================================")
    print(f"--- ENTRENANDO LIGHTGBM (PESOS NATIVOS) - {exp_name} ---")
    print("⏳ Por favor, espera. Calculando hojas (leaf-wise)...")
    
    # 1. INSTANCIACIÓN DEL MODELO
    # n_estimators=100 (Crea 100 iteraciones/árboles secuenciales)
    # class_weight='balanced' (Aplica el multiplicador estadístico automático a las averías)
    # n_jobs=-1 (VITAL: Usa todos los núcleos para paralelizar lo paralelizable)
    # verbose=-1 (Silencia las advertencias internas del motor en C++)
    model = lgb.LGBMClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced',
        n_jobs=-1,
        verbose=-1
    )
    
    # 2. ENTRENAMIENTO
    start_train = time.time()
    model.fit(X_train, y_train_enc)
    train_time = round(time.time() - start_train, 4)
    
    # 3. INFERENCIA
    start_inf = time.time()
    y_pred_enc = model.predict(X_val) # El modelo escupe números (0 al 6)
    inf_time_ms = round((time.time() - start_inf) * 1000, 2)
    
    # 4. DECODIFICACIÓN (Números -> Texto para la tabla)
    y_pred = label_encoder.inverse_transform(y_pred_enc)
    y_val = label_encoder.inverse_transform(y_val_enc)
    
    # 5. REPORTE COMPLETO
    print(f"✅ ¡Entrenamiento Finalizado!")
    print(f"[T. Entrenamiento: {train_time} seg | T. Inferencia: {inf_time_ms} ms]\n")
    print(classification_report(y_val, y_pred, zero_division=0))
    
    # 6. RETORNO DE VARIABLES (Para el Tracker)
    f1_macro = round(f1_score(y_val, y_pred, average='macro', zero_division=0), 4)
    report_dict = classification_report(y_val, y_pred, output_dict=True, zero_division=0)
    f1_minority = round(report_dict.get('Service Outages and Maintenance', {}).get('f1-score', 0), 4)
    
    return train_time, inf_time_ms, f1_macro, f1_minority


# EJECUTAMOS CONTROL (LIGHTGBM + PESOS, SIN SMOTE)
en_train_time_lgb_w, en_inf_time_lgb_w, en_f1_lgb_w, en_f1_min_lgb_w = train_evaluate_lgbm_weights(
    en_X_train, en_y_train_enc, en_X_val, en_y_val_enc, le, "INGLÉS (PESOS)"
)

es_train_time_lgb_w, es_inf_time_lgb_w, es_f1_lgb_w, es_f1_min_lgb_w = train_evaluate_lgbm_weights(
    es_X_train, es_y_train_enc, es_X_val, es_y_val_enc, le, "ESPAÑOL (PESOS)"
)


--- ENTRENANDO LIGHTGBM (PESOS NATIVOS) - INGLÉS (PESOS) ---
⏳ Por favor, espera. Calculando hojas (leaf-wise)...
✅ ¡Entrenamiento Finalizado!
[T. Entrenamiento: 36.1457 seg | T. Inferencia: 57.68 ms]

                                 precision    recall  f1-score   support

           Billing and Payments       0.82      0.80      0.81       381
               Customer Service       0.43      0.50      0.46       569
                     IT Support       0.46      0.46      0.46       451
                Product Support       0.46      0.45      0.45       701
            Sales and Pre-Sales       0.51      0.51      0.51       115
Service Outages and Maintenance       0.59      0.65      0.62       149
              Technical Support       0.63      0.58      0.60      1102

                       accuracy                           0.55      3468
                      macro avg       0.56      0.56      0.56      3468
                   weighted avg       0.55      0.55      0.55   

### Paso 3: Experimento B (LightGBM + Inyección Sintética SMOTE)

Una vez probado que el *Cost-Sensitive Learning* (Pesos de Clase) genera un rendimiento subóptimo en NLP, ejecutamos la evaluación final del *Boosting*. 
Retiramos los pesos matemáticos e inyectamos volumen físico en las clases minoritarias mediante la técnica SMOTE. El objetivo es comprobar si la combinación de *Leaf-wise Growth* (LightGBM) + *Data-level Balancing* (SMOTE) logra destronar al actual campeón del proyecto: Random Forest.

In [3]:
from imblearn.over_sampling import SMOTE

print("--- INICIANDO INYECCIÓN SINTÉTICA PARA LIGHTGBM ---")
# 1. INVOCAMOS A SMOTE (Directamente sobre las etiquetas numéricas)
smote = SMOTE(random_state=42)

start_smote = time.time()
en_X_train_smote, en_y_train_enc_smote = smote.fit_resample(en_X_train, en_y_train_enc)
es_X_train_smote, es_y_train_enc_smote = smote.fit_resample(es_X_train, es_y_train_enc)
print(f"✅ Matrices engordadas artificialmente en {round(time.time() - start_smote, 2)} seg.\n")


# 2. FUNCIÓN FÁBRICA LIGHTGBM (Sin Pesos, con SMOTE)
def train_evaluate_lgbm_smote(X_train_smote, y_train_enc_smote, X_val, y_val_enc, label_encoder, exp_name):
    print(f"=======================================================")
    print(f"--- ENTRENANDO LIGHTGBM (SMOTE) - {exp_name} ---")
    print("⏳ Entrenando sobre matriz hipertrofiada...")
    
    # INSTANCIACIÓN LIMPIA
    # Ya NO usamos class_weight='balanced', dejamos que aprenda por volumen de datos
    model = lgb.LGBMClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    
    # ENTRENAMIENTO (Sobre las matrices X e Y inyectadas por SMOTE)
    start_train = time.time()
    model.fit(X_train_smote, y_train_enc_smote)
    train_time = round(time.time() - start_train, 4)
    
    # INFERENCIA
    start_inf = time.time()
    y_pred_enc = model.predict(X_val)
    inf_time_ms = round((time.time() - start_inf) * 1000, 2)
    
    # DECODIFICACIÓN Y REPORTE
    y_pred = label_encoder.inverse_transform(y_pred_enc)
    y_val = label_encoder.inverse_transform(y_val_enc)
    
    print(f"✅ ¡Entrenamiento Finalizado!")
    print(f"[T. Entrenamiento: {train_time} seg | T. Inferencia: {inf_time_ms} ms]\n")
    print(classification_report(y_val, y_pred, zero_division=0))
    
    # RETORNO PARA TRACKER
    f1_macro = round(f1_score(y_val, y_pred, average='macro', zero_division=0), 4)
    report_dict = classification_report(y_val, y_pred, output_dict=True, zero_division=0)
    f1_minority = round(report_dict.get('Service Outages and Maintenance', {}).get('f1-score', 0), 4)
    
    return train_time, inf_time_ms, f1_macro, f1_minority


# 3. EJECUTAMOS EXPERIMENTO FINAL (LIGHTGBM + SMOTE)
en_train_lgb_sm, en_inf_lgb_sm, en_f1_lgb_sm, en_f1_min_lgb_sm = train_evaluate_lgbm_smote(
    en_X_train_smote, en_y_train_enc_smote, en_X_val, en_y_val_enc, le, "INGLÉS (SMOTE)"
)

es_train_lgb_sm, es_inf_lgb_sm, es_f1_lgb_sm, es_f1_min_lgb_sm = train_evaluate_lgbm_smote(
    es_X_train_smote, es_y_train_enc_smote, es_X_val, es_y_val_enc, le, "ESPAÑOL (SMOTE)"
)

--- INICIANDO INYECCIÓN SINTÉTICA PARA LIGHTGBM ---
✅ Matrices engordadas artificialmente en 1.07 seg.

--- ENTRENANDO LIGHTGBM (SMOTE) - INGLÉS (SMOTE) ---
⏳ Entrenando sobre matriz hipertrofiada...
✅ ¡Entrenamiento Finalizado!
[T. Entrenamiento: 101.8492 seg | T. Inferencia: 52.62 ms]

                                 precision    recall  f1-score   support

           Billing and Payments       0.87      0.76      0.81       381
               Customer Service       0.46      0.46      0.46       569
                     IT Support       0.51      0.36      0.42       451
                Product Support       0.47      0.45      0.46       701
            Sales and Pre-Sales       0.59      0.38      0.47       115
Service Outages and Maintenance       0.72      0.59      0.65       149
              Technical Support       0.57      0.71      0.63      1102

                       accuracy                           0.56      3468
                      macro avg       0.60      0.53

### Paso 4: Consolidación MLOps y Descarte Oficial del Boosting

Volcamos los 4 modelos de LightGBM en los Trackers Maestros. 
Los resultados empíricos certifican que, aunque LightGBM es un 150% más rápido que XGBoost gracias a su arquitectura *leaf-wise*, sigue sufriendo una severa penalización predictiva (F1 ~0.56) frente a Random Forest (F1 ~0.68) en espacios vectoriales altamente dispersos (NLP puro).

**Veredicto Final de Fase:** Quedan descartadas las arquitecturas de *Gradient Boosting*. **Random Forest + SMOTE** se corona como el Campeón Absoluto del proyecto y será el único algoritmo que avance a la Fase de Optimización de Hiperparámetros (GridSearchCV).

In [4]:
import pandas as pd

print("--- GUARDANDO EXPERIMENTOS LIGHTGBM EN EL TRACKER CENTRAL ---")

# 1. CARGAMOS LOS TRACKERS FÍSICOS
tracker_en = pd.read_csv("../data/processed/tracker_en.csv")
tracker_es = pd.read_csv("../data/processed/tracker_es.csv")

# 2. EMPAQUETAMOS (INGLÉS)
nuevos_experimentos_en = [
    {
        'exp_id': 'EN_L1_TFIDF_LGBM_WEIGHTS',
        'target_level': 'queue',
        'vectorization': 'tfidf',
        'model': 'lightgbm',
        'balancing': 'weights', 
        'hyperparameters': 'n_estimators=100, class_weight=balanced',
        'train_time_sec': en_train_time_lgb_w,
        'inference_time_ms': en_inf_time_lgb_w,
        'f1_macro': en_f1_lgb_w,
        'f1_minority_class': en_f1_min_lgb_w
    },
    {
        'exp_id': 'EN_L1_TFIDF_LGBM_SMOTE',
        'target_level': 'queue',
        'vectorization': 'tfidf',
        'model': 'lightgbm',
        'balancing': 'smote',
        'hyperparameters': 'n_estimators=100',
        'train_time_sec': en_train_lgb_sm,
        'inference_time_ms': en_inf_lgb_sm,
        'f1_macro': en_f1_lgb_sm,
        'f1_minority_class': en_f1_min_lgb_sm
    }
]

# 3. EMPAQUETAMOS (ESPAÑOL)
nuevos_experimentos_es = [
    {
        'exp_id': 'ES_L1_TFIDF_LGBM_WEIGHTS',
        'target_level': 'queue',
        'vectorization': 'tfidf',
        'model': 'lightgbm',
        'balancing': 'weights',
        'hyperparameters': 'n_estimators=100, class_weight=balanced',
        'train_time_sec': es_train_time_lgb_w,
        'inference_time_ms': es_inf_time_lgb_w,
        'f1_macro': es_f1_lgb_w,
        'f1_minority_class': es_f1_min_lgb_w
    },
    {
        'exp_id': 'ES_L1_TFIDF_LGBM_SMOTE',
        'target_level': 'queue',
        'vectorization': 'tfidf',
        'model': 'lightgbm',
        'balancing': 'smote',
        'hyperparameters': 'n_estimators=100',
        'train_time_sec': es_train_lgb_sm,
        'inference_time_ms': es_inf_lgb_sm,
        'f1_macro': es_f1_lgb_sm,
        'f1_minority_class': es_f1_min_lgb_sm
    }
]

# 4. INYECTAMOS Y SOBREESCRIBIMOS
tracker_en = pd.concat([tracker_en, pd.DataFrame(nuevos_experimentos_en)], ignore_index=True)
tracker_en.to_csv("../data/processed/tracker_en.csv", index=False)

tracker_es = pd.concat([tracker_es, pd.DataFrame(nuevos_experimentos_es)], ignore_index=True)
tracker_es.to_csv("../data/processed/tracker_es.csv", index=False)

print("✅ Experimentos de LightGBM registrados. ¡FASE DE BARRIDO COMPLETADA!")

# 5. AUDITORÍA VISUAL DEL TRACKER
print("\n--- TRACKER MAESTRO ACTUALIZADO (INGLÉS) ---")
display(tracker_en.tail(6))  # Mostramos RF, XGB y LGBM para la comparativa definitiva
print("\n--- TRACKER MAESTRO ACTUALIZADO (ESPAÑOL) ---")
display(tracker_es.tail(6))

--- GUARDANDO EXPERIMENTOS LIGHTGBM EN EL TRACKER CENTRAL ---
✅ Experimentos de LightGBM registrados. ¡FASE DE BARRIDO COMPLETADA!

--- TRACKER MAESTRO ACTUALIZADO (INGLÉS) ---


,exp_id,target_level,vectorization,model,balancing,train_time_sec,inference_time_ms,f1_macro,f1_minority_class,hyperparameters
3,EN_L1_TFIDF_RF_NONE,queue,tfidf,random_forest,none,4.9768,53.20,0.6128,0.6891,n_estimators=100
4,EN_L1_TFIDF_RF_SMOTE,queue,tfidf,random_forest,smote,15.6008,65.79,0.6849,0.7266,n_estimators=100
5,EN_L1_TFIDF_XGB_WEIGHTS,queue,tfidf,xgboost,weights,100.0310,104.37,0.5277,0.5465,"n_estimators=100, class_weight=balanced"
6,EN_L1_TFIDF_XGB_SMOTE,queue,tfidf,xgboost,smote,267.8697,81.67,0.5171,0.6124,n_estimators=100
7,EN_L1_TFIDF_LGBM_WEIGHTS,queue,tfidf,lightgbm,weights,36.1457,57.68,0.5582,0.6198,"n_estimators=100, class_weight=balanced"
8,EN_L1_TFIDF_LGBM_SMOTE,queue,tfidf,lightgbm,smote,101.8492,52.62,0.5567,0.6471,n_estimators=100



--- TRACKER MAESTRO ACTUALIZADO (ESPAÑOL) ---


,exp_id,target_level,vectorization,model,balancing,train_time_sec,inference_time_ms,f1_macro,f1_minority_class,hyperparameters
3,ES_L1_TFIDF_RF_NONE,queue,tfidf,random_forest,none,4.8221,62.61,0.5816,0.6121,n_estimators=100
4,ES_L1_TFIDF_RF_SMOTE,queue,tfidf,random_forest,smote,15.7321,65.97,0.6502,0.7000,n_estimators=100
5,ES_L1_TFIDF_XGB_WEIGHTS,queue,tfidf,xgboost,weights,95.1703,87.80,0.5030,0.5897,"n_estimators=100, class_weight=balanced"
6,ES_L1_TFIDF_XGB_SMOTE,queue,tfidf,xgboost,smote,234.1503,83.01,0.4941,0.5940,n_estimators=100
7,ES_L1_TFIDF_LGBM_WEIGHTS,queue,tfidf,lightgbm,weights,36.1125,60.31,0.5248,0.5919,"n_estimators=100, class_weight=balanced"
8,ES_L1_TFIDF_LGBM_SMOTE,queue,tfidf,lightgbm,smote,101.6787,51.96,0.5434,0.6022,n_estimators=100
